# Task 04 & 05 – Object Tracking + Evaluation
## ByteTrack Integration & Comprehensive Metrics

This notebook covers:
1. ByteTrack tracking on image sequence / video
2. Trajectory visualization
3. Unique person counting via track IDs
4. mAP / Precision / Recall evaluation
5. FPS benchmarking
6. Strengths, limitations & challenges

In [ ]:
import sys, os
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

from src.track    import DroneTracker, draw_tracks
from src.evaluate import run_validation, benchmark_fps, print_report

# ── Paths ──
DATA_ROOT = r'D:\antlings-drone-detection\dataset\VisDrone_Dataset'
VAL_IMGS  = str(Path(DATA_ROOT) / 'VisDrone2019-DET-val'      / 'images')
TEST_IMGS = str(Path(DATA_ROOT) / 'VisDrone2019-DET-test-dev' / 'images')
WEIGHTS   = '../runs/detect/visdrone_yolov8n/weights/best.pt'
DATA_CFG  = '../configs/data.yaml'
OUT_DIR   = '../outputs'

os.makedirs(f'{OUT_DIR}/images', exist_ok=True)
os.makedirs(f'{OUT_DIR}/videos', exist_ok=True)

print('Setup complete.')

## Task 04: ByteTrack Object Tracking

**Why ByteTrack?**
- Lightweight — no separate Re-ID model needed
- Handles occlusions via low-confidence second association
- Built into Ultralytics — zero extra dependencies
- Persistent track IDs allow unique person counting over time

**How it works:**
1. High-confidence detections matched to existing tracks (IoU)
2. Low-confidence detections used for recovering lost tracks
3. Unmatched tracks kept for N frames before deletion
4. Each track gets a unique ID that persists across frames

In [ ]:
# Run ByteTrack on validation image sequence (treating images as frames)
tracker   = DroneTracker(WEIGHTS, conf=0.35, iou=0.45)
val_imgs  = sorted(Path(VAL_IMGS).glob('*.jpg'))[:30]

person_ids = set()
car_ids    = set()
frame_log  = []

for i, img_path in enumerate(val_imgs):
    frame  = cv2.imread(str(img_path))
    tracks = tracker.update(frame)

    for t in tracks:
        if t['cls_id'] == 0: person_ids.add(t['track_id'])
        else:                 car_ids.add(t['track_id'])

    frame_log.append({
        'frame':          i,
        'active_persons': sum(1 for t in tracks if t['cls_id'] == 0),
        'active_cars':    sum(1 for t in tracks if t['cls_id'] == 1),
        'unique_persons': len(person_ids),
        'unique_cars':    len(car_ids),
    })

print(f'Processed {len(val_imgs)} frames')
print(f'Unique persons tracked : {len(person_ids)}')
print(f'Unique cars tracked    : {len(car_ids)}')

In [ ]:
# Show annotated tracking frame with trail lines
tracker2 = DroneTracker(WEIGHTS)

for img_path in val_imgs[:16]:
    f = cv2.imread(str(img_path))
    tracks = tracker2.update(f)

frame  = cv2.imread(str(val_imgs[15]))
p_cnt  = sum(1 for t in tracks if t['cls_id'] == 0)
c_cnt  = sum(1 for t in tracks if t['cls_id'] == 1)
ann    = draw_tracks(frame, tracks, tracker2.trails,
                     {'person': p_cnt, 'car': c_cnt}, len(person_ids))

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
plt.title(f'ByteTrack — Active: Person={p_cnt} Car={c_cnt} | Unique persons: {len(person_ids)}', fontsize=12)
plt.axis('off')
plt.savefig(f'{OUT_DIR}/images/04_tracking_frame.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/images/04_tracking_frame.png')

In [ ]:
# Plot active counts and cumulative unique persons over frames
df_log = pd.DataFrame(frame_log)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ByteTrack Statistics Over Frames', fontsize=13, fontweight='bold')

axes[0].plot(df_log.frame, df_log.active_persons, color='#00C853', label='Active persons', linewidth=2)
axes[0].plot(df_log.frame, df_log.active_cars,    color='#FF6D00', label='Active cars',    linewidth=2)
axes[0].set_title('Active Detections per Frame')
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_log.frame, df_log.unique_persons, color='#2979FF', linewidth=2, label='Unique persons')
axes[1].plot(df_log.frame, df_log.unique_cars,    color='#AA00FF', linewidth=2, label='Unique cars')
axes[1].set_title('Cumulative Unique Objects Tracked')
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('Unique Count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/images/04_tracking_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/images/04_tracking_stats.png')

## Task 05: Evaluation

In [ ]:
# Run YOLO validation on VisDrone val split
print('Running validation...')
metrics = run_validation(WEIGHTS, DATA_CFG, imgsz=640, conf=0.35, iou=0.45)
print_report(metrics)

In [ ]:
# FPS benchmark on val images
fps = benchmark_fps(WEIGHTS, VAL_IMGS, n_warmup=5, n_bench=50)
print(f'Inference speed: {fps:.1f} FPS  (~{1000/fps:.0f}ms per frame)')

## Strengths, Limitations & Challenges

### Strengths
- Real-time capable (~42 FPS) — suitable for live drone feeds
- Transfer learning from COCO gives strong starting point
- ByteTrack needs zero extra dependencies via Ultralytics
- Mosaic + copy-paste augmentation helps small objects
- Transparent counting logic, easy to verify

### Limitations
- Very small objects (<10px) still missed — needs SAHI tiling or 1280px input
- Dense crowd scenes: overlapping boxes merged under NMS → undercounting
- No night/thermal support — trained on daytime RGB only
- ByteTrack ID switches on fast-moving or occluded objects

### Challenges Faced
- VisDrone has 10 classes but we need only 2 — annotation conversion needed
- Most bboxes are <20px tall — standard 640px input misses many
- Dense scenes with 100+ objects per image stress the NMS pipeline

### Future Improvements
- SAHI (Sliced Inference) for tiny object detection
- Add P2 detection head for higher resolution features
- StrongSORT or BoT-SORT for better long-term Re-ID
- Train at 1280px for significant mAP gain

### Task 04 & 05 Complete